In [1]:
%pip install python-dotenv

In [2]:
import os
from getpass import getpass

# This will pop up an input box at the top of VS Code when you run the cell
os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ_API_KEY: ")

print("Key loaded successfully!")

In [3]:
%pip install -q langchain langchain-core langchain-groq \
             langchain-community gradio pydantic

#### Runnable Lambda

In [4]:
from langchain_core.runnables import RunnableLambda
def yield_example(x):
    yield x**2
    yield x**3

double = RunnableLambda(yield_example)
print(double.invoke(2))
print(double.batch([3, 2]))
for data in double.stream(2):
    print(data)

In [5]:
from functools import partial
from langchain_core.runnables import RunnableLambda

def greet(name, greeting="hello"):
    return f"{greeting}, {name}"

formal_greet = RunnableLambda(partial(greet, greeting="Good day"))
casual_greet = RunnableLambda(partial(greet, greeting="Hey"))

print(formal_greet.invoke("arham"))
print(casual_greet.invoke("arham"))

In [6]:
from langchain_core.runnables import RunnableLambda

add_one = RunnableLambda(lambda x: x + 1)
multipy_by2 = RunnableLambda(lambda x: x*2)
to_string = RunnableLambda(lambda x: f"Result: {x}")

chain = add_one | multipy_by2 | to_string
print(chain.invoke(3))

In [7]:
from langchain_core.runnables import RunnableLambda
from functools import partial

def RPrint(preface="State "):
    def print_and_return(x, preface=""):
        print(f"{preface}{x}")
        return x
    return RunnableLambda(partial(print_and_return,preface=preface))


def printableState(runnable):
    return runnable | RPrint()

chain = printableState(add_one) | printableState(multipy_by2) | printableState(to_string)
chain.invoke(2)

Dictionary type in input to runnables

In [8]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough, RunnableParallel

pipeline1 = RunnableParallel({
    'first_word': RunnableLambda(lambda x: x.split()[0]),
    'second_word': RunnableLambda(lambda x: x.split()[1]),
    'full': RunnablePassthrough()
})

def print_data(x):
    return f"end data: {x}"

chain = pipeline1 | print_data

print(chain.invoke("hello world"))

### LLM

Chat Prompt Template

In [9]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant with name {name}, always answer in rhymes"),
    ("user", "{input}")
])

result = prompt.invoke({"name": "gimy", "input": "explain the meaning of your name" })
print(type(result))
print(result.messages)

In [10]:
prompt = ChatPromptTemplate.from_template(
    "Classify this sentence into one category.\n"
    "Options: {options}\n\n"
    "Sentence: {input}"
)
prompt.invoke({"input": "I love sailing", "options": "boat, car, plane"})
print(prompt.messages)

LLM

In [11]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant")

response = llm.invoke("who is mark zuckerburg")
print(type(response))
print(response.content)

### Full Simple Chain

In [12]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGroq(model='llama-3.3-70b-versatile')
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a comedian, only speak with great humor"),
    ("user", "{input}")
])
parser = StrOutputParser()

chain = prompt | llm | parser
print(chain.invoke({"input": "Tell me about Russia"}))

for token in chain.stream({"input": "tell me about russia"}):
    print(token, end='', flush=True)

# Internal vs External Reasoning

### Zero Shot Classification

In [13]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

instruct_llm = ChatGroq(model="llama-3.1-8b-instant")
one_word_llm = instruct_llm.bind(stop=[" ", "\n"]) | StrOutputParser()

zsc_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Choose the most likely topic category for the sentence below. "
     "Options: {options}. "
     "Respond with exactly ONE word from the options. No explanation."),
    ("user", "{input}")]
)

zsc_chain = zsc_prompt | one_word_llm

def classify(text, options=["car", "boat", "airplane", "bike"]):
    output = zsc_chain.invoke({"input": text, "options": ", ".join(options)})

    words = output.strip().split()
    return words[0]

print(f"Result: {classify('Should i take the next exit')}")

## Multi-Component Chain

In [14]:
# Runnable Assign
from langchain_core.runnables import RunnableAssign, RunnableLambda

add_ten = RunnableLambda(lambda state: state["value"] + 10)
chain = RunnableAssign({"result": add_ten})
output = chain.invoke({"value": 5, "label": "My number"})
print(output)

In [15]:
from langchain_core.runnables import RunnableAssign, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

instruct_llm = ChatGroq(model="llama-3.1-8b-instant")
one_word_llm = instruct_llm.bind(stop=[" ", "\n"]) | StrOutputParser()

zsc_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Choose the most likely topic category for the sentence below. "
     "Options: {options}. "
     "Respond with exactly ONE word from the options. No explanation."),
    ("user", "{input}")]
)

zsc_chain = zsc_prompt | one_word_llm

llm2 = ChatGroq(model="llama-3.1-8b-instant")
instruction = ChatPromptTemplate.from_template(
    "From the given word {response} please make me a sentence related to that"
)
chain2 = instruction | llm2 | StrOutputParser()

chain = zsc_chain | RunnableLambda(lambda word: {"response": word}) | chain2
chain.invoke({"input": "I like driving", "options": ["plane", "car", "food"]})

In [16]:
from langchain_core.runnables import RunnableAssign, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter


instruct_llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0)

one_word_llm = instruct_llm.bind(stop=[" ", "\n"]) | StrOutputParser()

zsc_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Choose the most likely topic category for the sentence below. "
     "Options: {options}. "
     "Respond with exactly ONE word from the options. No explanation."),
    ("user", "{input}")
])

zsc_chain = zsc_prompt | one_word_llm

sentence_instruction = ChatPromptTemplate.from_template(
    "From the given word {response} please make me a sentence related to that"
)

llm2 = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7)

chain1 = RunnableAssign({
    "response": zsc_chain | (lambda x: x.strip().split()[0])
})

chain2 = RunnableAssign({
    "result": sentence_instruction | llm2 | StrOutputParser()
})

chain = chain1 | chain2

chain.invoke({"input": "I like big things", "options": ["airplane", "earth", "universe", "sun", "car"]})

## Running State Chains

In [17]:
from pydantic import BaseModel, Field
from typing import Optional, List, Dict, Any

class KnowledgeBase(BaseModel):
    topic: str = Field('general', description="Current conversation topic")
    user_preference: Dict[str, Any] = Field({}, description="User preference as key-value pairs")
    session_notes: List[str] = Field([], description="Notes about this session")
    unresolved_queries: List[str] = Field([], description="Questions not yet answered")

kb = KnowledgeBase()
print(kb.topic)
print(kb.session_notes)


In [18]:
kb = KnowledgeBase(topic="Travel", session_notes=["User likes beaches"])
print(repr(kb))
print(kb.model_dump())

In [19]:
from pydantic import BaseModel, Field
from typing import Optional

class KnowledgeBase(BaseModel):
    first_name:          str          = Field('unknown', description="User's first name, `unknown` if not yet mentioned")
    last_name:           str          = Field('unknown', description="User's last name, `unknown` if not yet mentioned")
    confirmation:        Optional[int]= Field(None,      description="Flight confirmation number as integer, None if unknown")
    discussion_summary:  str          = Field("",        description="Running summary of the full conversation so far")
    open_problems:       str          = Field("",        description="Issues raised but not yet resolved")
    current_goals:       str          = Field("",        description="What the user is trying to accomplish right now")

In [20]:
from langchain_core.output_parsers import PydanticOutputParser
parser = PydanticOutputParser(pydantic_object=KnowledgeBase)

# instructions = parser.get_format_instructions()
# print(instructions)

# parsing a string
result = parser.parse('{"first_name": "Jane", "last_name": "Doe", "confirmation": 12345}')
print(type(result))
print(result.first_name)
print(result.confirmation)

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser 
from langchain_core.runnables import RunnableAssign

def RExtract(pydantic_class, llm, prompt):
    parser = PydanticOutputParser(pydantic_object = pydantic_class)

    # adding instructions to the prompt
    instruct_merge = RunnableAssign({'format_instructions': lambda x: parser.get_format_instructions()})

    def preparse(string):
        if '{' not in string: string = '{' + string
        if '}' not in string: string = '}' + string
        string = (string
                        .replace("\\_", '_')
                        .replace("\n", " ")
                        .replace("\]", "]")
                        .replace("\[", "[")
                        )
        return string

    return instruct_merge | prompt | llm | preparse | parser


In [ ]:
from langchain_groq import ChatGroq